# Nestlé WISER DOM experiment laboratory

This notebook builds evidence before any report or presentation is generated.
It uses five canonical runtime CSV inputs and, when present, two optional
recommendation-output CSVs. Only aggregate metrics and figures are saved.

Notebook evidence is isolated under results/challenge-study/notebook; the command-line
study writes to results/challenge-study/cli. Start with the smoke profile to check
installation. Set NESTLE_EXPERIMENT_PROFILE=full for final challenge evidence. Every
experiment is checkpointed separately, so a long run can resume without repeating
completed studies.


In [ ]:
from __future__ import annotations

import os
from dataclasses import asdict
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

from domopt.checkpoints import (
    StaleCheckpointError,
    challenge_results_root,
    checkpoint_identity,
    checkpoint_run_directory,
    load_checkpoint,
    write_checkpoint,
)
from domopt.experiments import (
    experiment_profile,
    make_ibm_hardware_study_problem,
    rank_ibm_hardware_strategies,
    run_challenge_experiments,
    run_ibm_hardware_study,
    write_experiment_results,
)
from domopt.hardware import (
    benchmark_qubo_batch_scoring,
    discover_ibm_backends,
    hardware_capabilities,
)
from domopt.poc import (
    POC_REFERENCE_FILENAMES,
    PocConfig,
    audit_poc_bundle,
    audit_poc_outputs,
    load_poc_problem,
    prune_pareto_candidates,
)
from domopt.visualization import (
    plot_challenge_results,
    plot_hardware_benchmark,
    plot_ibm_backend_snapshot,
    plot_ibm_hardware_study,
)


def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (
            (candidate / "pyproject.toml").is_file()
            and (candidate / "src/domopt").is_dir()
        ):
            return candidate
    raise RuntimeError(
        "Run this notebook from inside the wiser-dom-optimization repository"
    )


PROJECT_ROOT = find_project_root(Path.cwd())
BUNDLE_DIR = Path(
    os.environ.get(
        "NESTLE_BUNDLE_DIR",
        PROJECT_ROOT / "data/raw/nestle_challenge",
    )
).expanduser().resolve()
PROFILE = os.environ.get(
    "NESTLE_EXPERIMENT_PROFILE", "smoke"
).strip().lower()
FORCE_RERUN = os.environ.get("NESTLE_FORCE_RERUN", "0") == "1"
ENABLE_GPU_BENCHMARK = (
    os.environ.get("DOMOPT_ENABLE_GPU_BENCHMARK", "0") == "1"
)
ENABLE_IBM_HARDWARE = (
    os.environ.get("DOMOPT_ENABLE_IBM_QPU", "0") == "1"
)
IBM_HARDWARE_PROFILE = os.environ.get(
    "DOMOPT_IBM_HARDWARE_PROFILE", "quick"
).strip().lower()
IBM_SHOTS = int(os.environ.get("DOMOPT_IBM_SHOTS", "512"))
IBM_BACKEND_NAME = os.environ.get("DOMOPT_IBM_BACKEND") or None
OUTPUT_ROOT = challenge_results_root(PROJECT_ROOT, producer="notebook")

print(
    {
        "project_root": str(PROJECT_ROOT),
        "bundle_dir": str(BUNDLE_DIR),
        "profile": PROFILE,
        "force_rerun": FORCE_RERUN,
        "notebook_results_root": str(OUTPUT_ROOT),
        "ibm_hardware_profile": IBM_HARDWARE_PROFILE,
        "requested_ibm_backend": IBM_BACKEND_NAME or "least_busy",
    }
)


## 1. Runtime-input readability gate

The optimizer needs only five tables: orders, inventory planning, shipping
lanes, dock capacity, and throughput observations. The challenge PDF,
equations document, and example workbook explain the task but are not solver
inputs. The two recommendation outputs are optional audit references.

If downloads contain names such as input_order data(1).csv, run
scripts/prepare_challenge_bundle.py first. It creates stable names and excludes
numbered duplicates and macOS metadata files.


In [ ]:
file_audit = audit_poc_bundle(BUNDLE_DIR)
assert file_audit["readable"].all()
display(
    file_audit[
        ["role", "filename", "rows", "columns", "readable"]
    ]
)


## 2. Build and audit the real POC model

This step converts planning units to integer cases, identifies focus loads,
creates eligible DC/date options, protects five days of inventory, and applies
documented dock and penalty rules. Pareto pruning is delayed so its effect can
be measured rather than assumed.


In [ ]:
problem_unpruned = load_poc_problem(
    BUNDLE_DIR,
    config=PocConfig(pareto_prune=False),
    strict_bundle_audit=False,
)
problem_pruned = prune_pareto_candidates(problem_unpruned)
summary = pd.DataFrame(
    [
        {
            "variant": "unpruned",
            "orders": len(problem_unpruned.orders),
            "assignment_groups": problem_unpruned.orders[
                "assignment_group"
            ].nunique(),
            "order_lines": len(problem_unpruned.order_lines),
            "candidate_rows": len(problem_unpruned.candidates),
        },
        {
            "variant": "pareto_pruned",
            "orders": len(problem_pruned.orders),
            "assignment_groups": problem_pruned.orders[
                "assignment_group"
            ].nunique(),
            "order_lines": len(problem_pruned.order_lines),
            "candidate_rows": len(problem_pruned.candidates),
        },
    ]
)
display(summary)

reference_available = all(
    (BUNDLE_DIR / name).is_file()
    for name in POC_REFERENCE_FILENAMES.values()
)
if reference_available:
    display(
        pd.Series(
            audit_poc_outputs(BUNDLE_DIR, problem_unpruned),
            name="reference audit",
        )
    )
else:
    print(
        "Optional recommendation outputs are absent; "
        "reconciliation is skipped."
    )


## 3. Content-addressed experiment runner

Each following cell runs one study and writes its aggregate table immediately.
A checkpoint is reusable only when its profile, complete configuration, problem
and bundle fingerprints, schema and assumption versions, objective version,
commit, dirty-source hash, columns, and row count match. Stale or incomplete
artifacts are rejected and recomputed. Set NESTLE_FORCE_RERUN=1 to bypass every
valid checkpoint. Feasibility is independently checked before evidence is used.


In [ ]:
experiment_frames: dict[str, pd.DataFrame] = {}
PROFILE_SETTINGS = experiment_profile(PROFILE)
PROFILE_CONFIGURATION = asdict(PROFILE_SETTINGS)
SUITE_IDENTITY = checkpoint_identity(
    problem_unpruned,
    profile=PROFILE,
    experiment="challenge_suite",
    configuration=PROFILE_CONFIGURATION,
)
OUTPUT_DIR = checkpoint_run_directory(OUTPUT_ROOT, SUITE_IDENTITY)
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print(f"artifact scope: {OUTPUT_DIR.relative_to(PROJECT_ROOT)}")


def feasible_mask(frame: pd.DataFrame) -> pd.Series:
    values = frame["feasible"]
    if values.dtype == bool:
        return values
    return values.astype(str).str.lower().isin({"true", "1"})


def run_or_load(name: str) -> pd.DataFrame:
    identity = checkpoint_identity(
        problem_unpruned,
        profile=PROFILE,
        experiment=name,
        configuration=PROFILE_CONFIGURATION,
    )
    path = TABLE_DIR / f"{name}.csv"
    frame = None
    if not FORCE_RERUN:
        try:
            frame = load_checkpoint(path, identity)
            print(f"loaded verified checkpoint: {path.relative_to(PROJECT_ROOT)}")
        except StaleCheckpointError as error:
            print(f"checkpoint unavailable or stale ({error}); recomputing {name}")
    if frame is None:
        frame = run_challenge_experiments(
            problem_unpruned,
            profile=PROFILE_SETTINGS,
            experiments=[name],
        )
        write_experiment_results(frame, path)
        write_checkpoint(frame, path, identity)
        print(f"wrote verified checkpoint: {path.relative_to(PROJECT_ROOT)}")

    invalid = frame.loc[~feasible_mask(frame)]
    if not invalid.empty:
        diagnostic_columns = [
            "experiment",
            "level",
            "validation_categories",
            "validation_violation_count",
            "error_type",
        ]
        columns = [column for column in diagnostic_columns if column in invalid]
        raise RuntimeError(
            "Infeasible experiment rows: "
            f"{invalid[columns].to_dict('records')}"
        )
    experiment_frames[name] = frame
    return frame


## 4. Common solver comparison

This is the central fairness test. Default routing, load-atomic greedy, polished
greedy, adaptive exact-MILP LNS, full exact MILP, and sampler-assisted LNS use
the same business objective and independent validator. Polished greedy isolates
quantity-recourse gains. Exact LNS leaves local assignments and quantities free
in one joint MILP, so its assignment-search gain is distinct from that polish.
The raw objective is a source-currency total, so normalized capture is shown too.


In [ ]:
solver_results = run_or_load("solver_comparison")
display(
    solver_results[
        [
            "method",
            "feasible",
            "objective_value",
            "requested_value",
            "objective_capture_rate",
            "case_fill_rate",
            "penalty_cost",
            "shipping_cost",
            "runtime_seconds",
            "optimality_gap",
            "initial_polish_improvement",
            "search_improvement",
            "maximum_local_variables",
            "maximum_qubo_variables",
        ]
    ].sort_values("objective_value", ascending=False)
)


## 5. Real assignment-group size scaling

A load can contain several order records, so the true atomic decision count is
the number of assignment groups. The full profile repeats each measured point
three times for median timing. It scales greedy, polished greedy, and bounded
exact LNS through the real universe, bounds sampler-assisted LNS to tractable
QUBOs, and runs full MILP only where a useful certificate is realistic. Quality
uses objective capture because requested value grows with subset size.


In [ ]:
scaling_results = run_or_load("size_scaling")
display(
    scaling_results[
        [
            "method",
            "actual_assignment_groups",
            "repetition",
            "order_count",
            "order_line_count",
            "candidate_count",
            "maximum_local_variables",
            "maximum_qubo_variables",
            "objective_value",
            "objective_capture_rate",
            "case_fill_rate",
            "runtime_seconds",
            "feasible",
        ]
    ].sort_values(["actual_assignment_groups", "method", "repetition"])
)


## 6. Controlled synthetic scaling

Nested real subsets change both size and composition. This independent control
generates a new coupled instance for every size and repetition, then compares
greedy, exact LNS, sampler-assisted LNS where tractable, and full MILP on the
smallest cases. It supports complexity claims only, never real-business impact.


In [ ]:
synthetic_scaling_results = run_or_load("synthetic_scaling")
display(
    synthetic_scaling_results[
        [
            "method",
            "actual_assignment_groups",
            "repetition",
            "generator_seed",
            "candidate_count",
            "objective_capture_rate",
            "runtime_seconds",
            "feasible",
        ]
    ].sort_values(["actual_assignment_groups", "method", "repetition"])
)


## 7. Candidate-DC universe sensitivity

Candidate generation is a modeling assumption, not a harmless preprocessing
choice. This study compares the legacy focus/default-DC universe with every DC
in the shipping, inventory, and dock-capacity intersection. Identical results
are still useful negative evidence; different results quantify the restriction.


In [ ]:
candidate_scope_results = run_or_load(
    "candidate_dc_scope_sensitivity"
)
display(
    candidate_scope_results[
        [
            "candidate_dc_scope",
            "method",
            "candidate_count",
            "objective_capture_rate",
            "case_fill_rate",
            "runtime_seconds",
            "feasible",
        ]
    ].sort_values(["candidate_dc_scope", "method"])
)


## 8. Business penalty-weight sensitivity

Unmet-demand penalties encode the cost of poor service. This study uses a
fixed set of loads whose default fill is below the penalty threshold and ranks
them by active penalty exposure; otherwise a shortage-only sample can contain
zero-penalty orders and make the sweep meaningless. Scaling the penalties then
tests whether routing decisions are stable or driven by one arbitrary
coefficient. The important trade-off is fill and penalty reduction versus
extra shipping. Raw objectives across different penalty scales are not
directly comparable.


In [ ]:
business_penalty_results = run_or_load(
    "penalty_weight_sensitivity"
)
display(
    business_penalty_results[
        [
            "penalty_scale",
            "method",
            "case_fill_rate",
            "reassigned_orders",
            "penalty_cost",
            "shipping_cost",
            "runtime_seconds",
        ]
    ].sort_values(["penalty_scale", "method"])
)


## 9. QUBO penalty calibration

These are algorithmic penalties, not business costs. The one-hot multiplier
discourages selecting zero or multiple options for a load; the pair multiplier
discourages competing plans from overusing shared resources. The sweep
measures raw one-hot rate, repair burden, final improvement, and runtime so
penalties are tuned with evidence rather than guessed.


In [ ]:
qubo_penalty_results = run_or_load("qubo_penalty_sensitivity")
display(
    qubo_penalty_results[
        [
            "one_hot_penalty_multiplier",
            "pair_penalty_multiplier",
            "raw_one_hot_rate",
            "hybrid_improvement",
            "accepted_moves",
            "recourse_solves",
            "runtime_seconds",
        ]
    ].sort_values(
        ["one_hot_penalty_multiplier", "pair_penalty_multiplier"]
    )
)


## 10. Candidate-count sensitivity

Keeping more DC/date alternatives can improve the solution, but it increases
preprocessing, QUBO width, and recourse work. This experiment locates the point
where extra candidates stop paying for their computational cost.


In [ ]:
candidate_results = run_or_load("candidate_count_sensitivity")
display(
    candidate_results[
        [
            "candidate_limit",
            "method",
            "candidate_count",
            "maximum_qubo_variables",
            "objective_value",
            "case_fill_rate",
            "runtime_seconds",
        ]
    ].sort_values(["candidate_limit", "method"])
)


## 11. Inventory-shock robustness

The deterministic objective already protects projected ATP and charges shortage
penalties. A generic extra risk term would double-count risk without scenario
probabilities. This study distinguishes a nominal routing with exact quantity
recourse from policies fully reoptimized after observing each shock. That separates
fixed-policy robustness from wait-and-see recourse.


In [ ]:
shock_results = run_or_load("inventory_shock")
display(
    shock_results[
        [
            "inventory_shock",
            "method",
            "objective_value",
            "objective_capture_rate",
            "case_fill_rate",
            "unassigned_orders",
            "penalty_cost",
            "runtime_seconds",
        ]
    ].sort_values(["inventory_shock", "method"])
)


## 12. Seed and local QUBO coefficient-noise robustness

Repeated seeds test stochastic stability. Coefficient perturbations approximate
analog/control sensitivity in the local QUBO only; they are not a physical
device-noise model. Exact quantity recourse and final validation remain
unchanged, so weak samples cannot degrade the returned incumbent.


In [ ]:
noise_results = run_or_load("qubo_coefficient_noise")
display(
    noise_results[
        [
            "seed",
            "coefficient_noise_relative_sigma",
            "raw_one_hot_rate",
            "hybrid_improvement",
            "accepted_moves",
            "runtime_seconds",
        ]
    ].sort_values(["coefficient_noise_relative_sigma", "seed"])
)


## 13. Local QAOA readout-noise proxy

The ideal Dicke/XY circuit preserves one-hot feasibility. This separate control
applies independent symmetric bit flips only at measurement, before one-hot repair,
to measure how readout errors affect raw feasibility and validated improvement. It
is a physical measurement-channel proxy, not a gate, decoherence, or QPU model.


In [ ]:
readout_noise_results = run_or_load("qaoa_readout_noise")
display(
    readout_noise_results[
        [
            "seed",
            "qaoa_readout_bitflip_probability",
            "raw_one_hot_rate",
            "hybrid_improvement",
            "accepted_moves",
            "runtime_seconds",
            "feasible",
        ]
    ].sort_values(["qaoa_readout_bitflip_probability", "seed"])
)


## 14. Heuristic Pareto-pruning ablation

Isolated score dominance is not a globally lossless rule when options consume
different inventory or capacity buckets, so pruning is disabled by default.
This explicitly labeled ablation checks its observed speed/width benefit and
verifies whether it changes the validated solution on this instance.


In [ ]:
pruning_results = run_or_load("pareto_pruning_ablation")
display(
    pruning_results[
        [
            "level",
            "candidate_count",
            "maximum_qubo_variables",
            "hybrid_improvement",
            "runtime_seconds",
            "feasible",
        ]
    ]
)


## 15. Random versus conflict-based batches

A local hybrid move is useful only if its loads interact. Conflict batching
groups loads that compete for inventory or capacity; random batching is the
control. The comparison tests whether problem-aware decomposition finds better
moves under the same QUBO and runtime limits.


In [ ]:
batch_results = run_or_load("batch_strategy_ablation")
display(
    batch_results[
        [
            "level",
            "hybrid_improvement",
            "accepted_moves",
            "maximum_qubo_variables",
            "recourse_solves",
            "runtime_seconds",
        ]
    ]
)


## 16. Sampler and quantum-simulation ablation

A coupled synthetic control compares random and simulated annealing with exact
feasible enumeration and local gate-model QAOA. The QAOA state is a product of
weight-one Dicke/W states with XY ring mixers, so ideal samples are one-hot by
construction. Identical exact recourse isolates proposal quality.


In [ ]:
sampler_results = run_or_load("sampler_ablation")
display(
    sampler_results[
        [
            "level",
            "raw_one_hot_rate",
            "initial_polish_improvement",
            "hybrid_improvement",
            "accepted_moves",
            "quantum_simulator_calls",
            "runtime_seconds",
            "feasible",
        ]
    ]
)


## 17. Synthetic coordination control

Real-data subsets may not contain a case where local search beats greedy. This
independently generated control contains coupled choices that expose greedy
myopia. It tests the architecture, but it must be labeled synthetic and cannot
support a real-data or quantum-advantage claim.


In [ ]:
synthetic_results = run_or_load(
    "synthetic_coordination_control"
)
display(
    synthetic_results[
        [
            "method",
            "objective_value",
            "case_fill_rate",
            "runtime_seconds",
            "optimality_gap",
            "raw_initial_objective",
            "initial_polish_improvement",
            "hybrid_improvement",
            "total_hybrid_improvement",
            "feasible",
        ]
    ].sort_values("objective_value", ascending=False)
)


## 18. Persist all graphics

This cell combines every manifest-verified experiment, writes aggregate_results.csv
with its own content hash manifest, and saves stable PNG files in the same
profile/problem/source-scoped run directory. It displays each image inline. These
are evidence plots, not a final report or slide deck.


In [ ]:
prepared_frames = [
    frame.dropna(axis=1, how="all")
    for frame in experiment_frames.values()
]
results = pd.concat(prepared_frames, ignore_index=True, sort=False)
aggregate_identity = checkpoint_identity(
    problem_unpruned,
    profile=PROFILE,
    experiment="aggregate_results",
    configuration=PROFILE_CONFIGURATION,
)
aggregate_path = write_experiment_results(
    results, OUTPUT_DIR / "aggregate_results.csv"
)
write_checkpoint(results, aggregate_path, aggregate_identity)
figure_paths = plot_challenge_results(results, FIGURE_DIR)
print(
    f"wrote {len(results)} aggregate rows to "
    f"{aggregate_path.relative_to(PROJECT_ROOT)}"
)
for name, path in figure_paths.items():
    display(Markdown(f"### {name.replace('_', ' ').title()}"))
    display(Image(filename=str(path)))
print(
    f"saved {len(figure_paths)} figures in "
    f"{FIGURE_DIR.relative_to(PROJECT_ROOT)}"
)


## 19. Optional GPU crossover benchmark

The exact SciPy/HiGHS MILP is CPU-based, and the real bottlenecks are candidate
processing and exact recourse. Current local QUBOs are small, so GPU launch and
transfer overhead may exceed the work saved. This synthetic benchmark measures
only batched QUBO energy scoring and shows where an RTX-class GPU begins to
help. It does not claim end-to-end solver acceleration.

Set DOMOPT_ENABLE_GPU_BENCHMARK=1 and install the GPU extra on a CUDA machine
to run it. NVIDIA cuOpt can be evaluated later as an experimental MILP backend,
but it should be benchmarked against HiGHS before adoption.


In [ ]:
capabilities = hardware_capabilities()
display(pd.Series(capabilities, name="hardware capability"))
if ENABLE_GPU_BENCHMARK:
    hardware_results = benchmark_qubo_batch_scoring(
        include_gpu=True
    )
    hardware_path = OUTPUT_DIR / "hardware_qubo_scoring.csv"
    hardware_results.to_csv(hardware_path, index=False)
    hardware_figure = plot_hardware_benchmark(
        hardware_results,
        FIGURE_DIR / "hardware_qubo_scoring.png",
    )
    display(hardware_results)
    display(Image(filename=str(hardware_figure)))
else:
    print(
        "GPU benchmark skipped. "
        "Set DOMOPT_ENABLE_GPU_BENCHMARK=1 to run it."
    )


## 20. IBM backend discovery and hardware stress test

This opt-in study sends only an independently generated coupled control—never Nestlé
values or identifiers. It queries every accessible operational IBM QPU, records the
queue snapshot, and uses the least-busy eligible device unless DOMOPT_IBM_BACKEND is
set. The matched study compares p=1 and p=2 Dicke/XY-QAOA, baseline execution,
dynamical decoupling, and dynamical decoupling plus measurement twirling.

The table separates raw one-hot feasibility, exact-optimum hit rate, validated
assignment improvement, transpiled two-qubit cost, queue time, execution time, quantum
usage, and end-to-end runtime. Set DOMOPT_ENABLE_IBM_QPU=1 after configuring IBM
credentials. DOMOPT_IBM_HARDWARE_PROFILE=quick submits four jobs; `presentation`
repeats the same matrix over three seeds. This is a hardware-quality study, not a
quantum-advantage claim.


In [ ]:
if ENABLE_IBM_HARDWARE:
    backend_snapshot = discover_ibm_backends(min_num_qubits=16)
    selected_backend = IBM_BACKEND_NAME or str(
        backend_snapshot.loc[
            backend_snapshot["selected_least_busy"], "backend"
        ].iloc[0]
    )
    if selected_backend not in set(backend_snapshot["backend"].astype(str)):
        raise RuntimeError(
            f"Requested backend {selected_backend!r} is not eligible for this circuit"
        )
    backend_snapshot["selected_for_study"] = (
        backend_snapshot["backend"].astype(str).eq(selected_backend)
    )
    ibm_problem = make_ibm_hardware_study_problem()
    qpu_identity = checkpoint_identity(
        ibm_problem,
        profile=f"ibm-{IBM_HARDWARE_PROFILE}",
        experiment="ibm_hardware_stress",
        configuration={
            "backend": selected_backend,
            "shots": IBM_SHOTS,
            "hardware_profile": IBM_HARDWARE_PROFILE,
            "data_scope": "independently generated synthetic control",
        },
    )
    IBM_OUTPUT_DIR = checkpoint_run_directory(OUTPUT_ROOT, qpu_identity)
    IBM_TABLE_DIR = IBM_OUTPUT_DIR / "tables"
    IBM_FIGURE_DIR = IBM_OUTPUT_DIR / "figures"
    IBM_TABLE_DIR.mkdir(parents=True, exist_ok=True)
    IBM_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    backend_path = IBM_OUTPUT_DIR / "ibm_backend_snapshot.csv"
    backend_snapshot.to_csv(backend_path, index=False)
    backend_figure = plot_ibm_backend_snapshot(
        backend_snapshot,
        IBM_FIGURE_DIR / "ibm_backend_queue.png",
    )
    display(backend_snapshot)
    display(Image(filename=str(backend_figure)))

    qpu_target = IBM_TABLE_DIR / "ibm_hardware_stress.csv"
    qpu_results = run_ibm_hardware_study(
        allow_remote=True,
        backend_name=selected_backend,
        shots=IBM_SHOTS,
        profile=IBM_HARDWARE_PROFILE,
        progress_callback=lambda frame: write_experiment_results(
            frame, qpu_target
        ),
    )
    qpu_path = write_experiment_results(
        qpu_results,
        qpu_target,
    )
    write_checkpoint(qpu_results, qpu_path, qpu_identity)
    strategy_ranking = rank_ibm_hardware_strategies(qpu_results)
    ranking_path = write_experiment_results(
        strategy_ranking,
        IBM_TABLE_DIR / "ibm_strategy_ranking.csv",
    )
    qpu_figure = plot_ibm_hardware_study(
        qpu_results,
        IBM_FIGURE_DIR / "ibm_hardware_stress.png",
    )
    display(
        qpu_results[
            [
                "level",
                "hardware_backend",
                "hardware_mitigation_strategy",
                "qaoa_layers",
                "feasible",
                "search_improvement",
                "raw_one_hot_rate",
                "hardware_optimal_hit_rate",
                "hardware_transpiled_depth",
                "hardware_two_qubit_gates",
                "hardware_queue_seconds",
                "hardware_execution_seconds",
                "hardware_quantum_seconds",
                "hardware_returned_samples",
                "hardware_feasible_shots",
                "runtime_seconds",
            ]
        ]
    )
    display(strategy_ranking)
    best_observed = strategy_ranking.iloc[0]
    display(
        Markdown(
            f"**Best observed IBM strategy:** {best_observed['variant']} "
            f"(median raw optimum hit rate "
            f"{100 * best_observed['hardware_optimal_hit_rate']:.1f}%)."
        )
    )
    display(Image(filename=str(qpu_figure)))
    print(f"wrote {qpu_path.relative_to(PROJECT_ROOT)}")
    print(f"wrote {ranking_path.relative_to(PROJECT_ROOT)}")
else:
    print(
        "IBM hardware study disabled. Configure IBM Quantum and set "
        "DOMOPT_ENABLE_IBM_QPU=1."
    )


## 21. Interpretation guardrails and next evidence

A valid conclusion requires every returned solution to pass the independent
validator, hybrid never to fall below its greedy incumbent, and real POC
evidence to remain separate from synthetic controls. Compare business-penalty
settings through fill, penalty, shipping, and reassignment trade-offs rather
than raw objectives across different scales.

Do not add a generic risk coefficient until forecast scenarios and
probabilities can calibrate it. If those become available, the next defensible
extension is a scenario-based expected-shortfall or CVaR model compared with
the inventory-shock frontier here. Do not generate the final report or
presentation until the full profile and any approved hardware runs complete.
